# Dan 1 — EDA, deduplikacija, podela podataka

**Cilj danas:** razumeti podatke pre bilo kakvog treninga i napraviti pošten train/val/test split.

Redosled: Setup → EDA → Embeddings → Duplikati (**tvoj zadatak**) → Split → Bonus: prvi baseline za 1 minut.

> Kaggle podešavanja (desni panel → *Session options*): **Accelerator: GPU T4 x2 ili P100**, **Internet: On**.

## 0. Setup

In [ ]:
import os, sys, glob, json, subprocess
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle")
REPO_URL = "https://github.com/<TVOJ_USERNAME>/kp-dog-breeds.git"   # <-- promeni
WORK = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("NB_WORK", "work"))
WORK.mkdir(parents=True, exist_ok=True)

if ON_KAGGLE:
    repo = WORK / "kp-dog-breeds"
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=1.0.15", "cleanlab"], check=True)
    os.chdir(repo)
sys.path.insert(0, os.getcwd())

# Kaggle je menjao strukturu /kaggle/input -> ne pogađamo putanju, nego je nalazimo
if ON_KAGGLE:
    LABELS_CSV = Path(sorted(glob.glob("/kaggle/input/**/labels.csv", recursive=True))[0])
else:
    LABELS_CSV = Path(os.environ["NB_LABELS_CSV"])
IMAGES_DIR = LABELS_CSV.parent / "train"
print("labels:", LABELS_CSV)
print("slike: ", IMAGES_DIR, "| broj fajlova:", len(os.listdir(IMAGES_DIR)))

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 1. EDA — šta imamo?

Pitanja na koja tražiš odgovor (i zapisuješ u `docs/PLAN.md`):
1. Koliko slika po rasi? Da li je dataset neuravnotežen?
2. Kakve su rezolucije i odnosi stranica? (utiče na resize/crop)
3. Ima li grayscale/CMYK slika, oštećenih fajlova?
4. Kako slike izgledaju — pas u centru, više pasa, ljudi, tekst?

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

df = pd.read_csv(LABELS_CSV)
counts = df["breed"].value_counts()
print(f"{len(df)} slika, {df['breed'].nunique()} rasa")
print(f"po rasi: min {counts.min()} ({counts.idxmin()}), max {counts.max()} ({counts.idxmax()}), "
      f"medijana {counts.median():.0f}, odnos max/min = {counts.max()/counts.min():.1f}x")

fig, ax = plt.subplots(figsize=(18, 4))
counts.plot.bar(ax=ax, width=0.9)
ax.set_xticks([]); ax.set_ylabel("broj slika"); ax.set_title("Slika po rasi (sortirano)")
plt.show()

In [ ]:
# Metapodaci slika: PIL čita samo header -> brzo i za 10k fajlova
rows = []
for i in df["id"]:
    p = IMAGES_DIR / f"{i}.jpg"
    try:
        with Image.open(p) as im:
            rows.append({"id": i, "w": im.width, "h": im.height, "mode": im.mode, "kb": p.stat().st_size / 1024})
    except Exception as e:
        rows.append({"id": i, "w": None, "h": None, "mode": f"ERROR: {e}", "kb": None})
meta = pd.DataFrame(rows)
meta["aspect"] = meta["w"] / meta["h"]
print(meta[["w", "h", "aspect", "kb"]].describe().round(2))
print("\nmode:", meta["mode"].value_counts().to_dict())
print("slike manje od 224px po kraćoj ivici:", int((meta[["w", "h"]].min(axis=1) < 224).sum()))

fig, axs = plt.subplots(1, 2, figsize=(12, 3))
meta["aspect"].hist(bins=50, ax=axs[0]); axs[0].set_title("odnos stranica (w/h)")
meta[["w", "h"]].min(axis=1).hist(bins=50, ax=axs[1]); axs[1].set_title("kraća ivica (px)")
plt.show()

In [ ]:
# Pogledaj slike očima - 4 nasumične rase x 6 slika
rng = np.random.default_rng(0)
breeds = rng.choice(df["breed"].unique(), 4, replace=False)
fig, axs = plt.subplots(4, 6, figsize=(16, 11))
for r, b in enumerate(breeds):
    ids = df[df.breed == b]["id"].sample(6, random_state=0)
    for c, i in enumerate(ids):
        axs[r, c].imshow(Image.open(IMAGES_DIR / f"{i}.jpg").convert("RGB")); axs[r, c].axis("off")
    axs[r, 0].set_title(b, loc="left", fontsize=12)
plt.tight_layout(); plt.show()

## 2. Embeddings — "otisak" svake slike

Svaku sliku pretvaramo u vektor (768 brojeva) pomoću pretreniranog modela bez poslednjeg sloja (`num_classes=0`).
Slične slike → slični vektori. Koristimo **DINOv3 ConvNeXt-Tiny** (self-supervised, bez ImageNet labela).
Ove embeddinge koristimo 3 puta: deduplikacija (danas), brzi baseline (bonus), cleanlab (Dan 3).

In [ ]:
import timm
from torch.utils.data import DataLoader
from timm.data import resolve_data_config, create_transform
from dogbreeds.data import BreedDataset

EMB_MODEL = os.environ.get("NB_EMB_MODEL", "convnext_tiny.dinov3_lvd1689m")
PRETRAINED = os.environ.get("NB_PRETRAINED", "1") == "1"
emb_model = timm.create_model(EMB_MODEL, pretrained=PRETRAINED, num_classes=0).to(DEVICE).eval()
dc = resolve_data_config({}, model=emb_model)
classes = sorted(df["breed"].unique())

df.to_csv(WORK / "all.csv", index=False)
ds = BreedDataset(WORK / "all.csv", IMAGES_DIR, classes, create_transform(**dc, is_training=False))
dl = DataLoader(ds, batch_size=128, num_workers=2 if ON_KAGGLE else 0)

embs = []
with torch.no_grad(), torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
    for x, _ in dl:
        embs.append(emb_model(x.to(DEVICE)).float().cpu())
emb = torch.nn.functional.normalize(torch.cat(embs), dim=1)   # L2 normalizacija -> dot = kosinusna sličnost
np.save(WORK / "embeddings.npy", emb.numpy())
print("embeddings:", tuple(emb.shape))

## 3. Gotovo identične slike (isti pas, ista sesija slikanja)

Za svaku sliku tražimo najsličniju **drugu** sliku. Pogledaj histogram i parove pri vrhu,
pa **sam izaberi prag** iznad kog smatraš da je "isti pas / ista fotografija".

In [ ]:
E = emb.to(DEVICE)
n = E.shape[0]
nn_sim = torch.empty(n); nn_idx = torch.empty(n, dtype=torch.long)
pairs_all = []   # (i, j, sim) za sve parove iznad 0.85 - kandidati
for s in range(0, n, 1024):
    S = E[s:s + 1024] @ E.T                       # kosinusna sličnost blok x svi
    S[torch.arange(S.shape[0]), torch.arange(s, s + S.shape[0])] = -1   # isključi sebe
    v, ix = S.max(1); nn_sim[s:s + len(v)] = v.cpu(); nn_idx[s:s + len(v)] = ix.cpu()
    r, c = torch.where(S > 0.85)
    for a, b in zip((r + s).tolist(), c.tolist()):
        if a < b:
            pairs_all.append((a, b, float(S[a - s, b])))
pairs_all = pd.DataFrame(pairs_all, columns=["i", "j", "sim"]).sort_values("sim", ascending=False)

plt.figure(figsize=(10, 3)); plt.hist(nn_sim.numpy(), bins=100); plt.yscale("log")
plt.title("sličnost sa najbližim susedom"); plt.xlabel("kosinusna sličnost"); plt.show()
for t in [0.85, 0.90, 0.93, 0.95, 0.97, 0.99]:
    print(f"prag {t}: {(pairs_all.sim > t).sum():5d} parova")

In [ ]:
# Pogledaj parove na različitim nivoima sličnosti -> odatle biraš prag
def show_pairs(sub, title):
    k = len(sub)
    if k == 0: return
    fig, axs = plt.subplots(2, k, figsize=(3 * k, 6))
    axs = np.array(axs).reshape(2, k)
    for c, (_, r) in enumerate(sub.iterrows()):
        for row, idx in enumerate([int(r.i), int(r.j)]):
            axs[row, c].imshow(Image.open(IMAGES_DIR / f"{df.id[idx]}.jpg").convert("RGB")); axs[row, c].axis("off")
            axs[row, c].set_title(f"{df.breed[idx][:18]}" + (f"\nsim={r.sim:.3f}" if row == 0 else ""), fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

for lo, hi in [(0.97, 1.01), (0.93, 0.97), (0.88, 0.93)]:
    show_pairs(pairs_all[(pairs_all.sim >= lo) & (pairs_all.sim < hi)].head(6), f"sim u [{lo}, {hi})")

In [ ]:
# Zanimljivo: vrlo slični parovi sa RAZLIČITIM rasama = verovatno pogrešna labela (ili ista slika dva puta)
pairs_all["breed_i"] = df.breed.values[pairs_all.i]; pairs_all["breed_j"] = df.breed.values[pairs_all.j]
cross = pairs_all[(pairs_all.breed_i != pairs_all.breed_j) & (pairs_all.sim > 0.93)]
print(f"{len(cross)} vrlo sličnih parova sa različitim labelama")
show_pairs(cross.head(6), "isti pas, različita rasa?")

### 🧩 TVOJ ZADATAK: grupisanje duplikata (union-find)

Ako su A~B i B~C (iznad praga), onda A, B, C moraju biti u **istoj grupi** — čak i ako A i C nisu direktno slični.
To je problem **povezanih komponenti grafa**. Klasično rešenje: **Union-Find (Disjoint Set Union)**.

Implementiraj `build_groups(pairs, n)`:
- ulaz: lista parova `(i, j)` i broj slika `n`
- izlaz: niz dužine `n`, gde `groups[k]` = ID grupe slike `k` (slike bez duplikata imaju svoju grupu)

Hint: `parent = list(range(n))`; `find(x)` prati roditelje do korena (dodaj *path compression*); `union(a, b)` spaja korene.
Ako zapneš posle 20-30 minuta — pošalji mi šta imaš.

In [ ]:
THRESHOLD = 0.95   # <-- izaberi na osnovu slika iznad i obrazloži u dnevniku odluka

def build_groups(pairs, n):
    # TODO: implementiraj union-find. Trenutno: svaka slika je svoja grupa (bez efekta).
    return np.arange(n)

# --- test na malom primeru (ne menjaj) ---
g = build_groups([(0, 1), (1, 2), (4, 5)], 7)
ok = (g[0] == g[1] == g[2]) and (g[4] == g[5]) and len({g[0], g[3], g[4], g[6]}) == 4
print("✅ union-find radi" if ok else "❌ još nije implementirano / nije tačno")

In [ ]:
dup_pairs = pairs_all[pairs_all.sim > THRESHOLD][["i", "j"]].values.tolist()
df["group"] = build_groups(dup_pairs, len(df))
sizes = df["group"].value_counts()
print(f"prag {THRESHOLD}: {len(dup_pairs)} parova -> {int((sizes > 1).sum())} grupa sa duplikatima, "
      f"{int(sizes[sizes > 1].sum())} slika u njima, najveća grupa = {sizes.max()}")
LABELS_GROUPED = WORK / "labels_grouped.csv"
df[["id", "breed", "group"]].to_csv(LABELS_GROUPED, index=False)

## 4. Split: train / val / test

`dogbreeds.data` automatski koristi **grupni split** ako postoji kolona `group` — cela grupa ide u isti skup.

In [ ]:
from dogbreeds.config import load_config
from dogbreeds.data import prepare_splits

cfg = load_config("configs/baseline.yaml")
SPLITS = WORK / "splits"
stats = prepare_splits(str(LABELS_GROUPED), str(SPLITS), cfg["data"]["val_size"], cfg["data"]["test_size"], cfg["seed"])
print(json.dumps(stats, indent=2))

# provera: nijedna grupa ne sme biti u dva skupa
parts = {s: pd.read_csv(SPLITS / f"{s}.csv") for s in ["train", "val", "test"]}
leak = set(parts["train"].group) & (set(parts["val"].group) | set(parts["test"].group))
print("grupa koje cure iz train u val/test:", len(leak))

## 5. Bonus: prvi baseline za ~1 minut (linear probe na embeddinzima)

Logistička regresija na zamrznutim DINOv3 embeddinzima. Ovo je **donja granica** koju fine-tuning sutra mora da pobedi.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, top_k_accuracy_score

idx = {r.id: k for k, r in df.iterrows()}
X = emb.numpy(); y = df.breed.map({c: i for i, c in enumerate(classes)}).values
tr = [idx[i] for i in parts["train"].id]; va = [idx[i] for i in parts["val"].id]
clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(X[tr], y[tr])
pv = clf.predict_proba(X[va])
labels = clf.classes_
print(f"linear probe ({EMB_MODEL}): val top-1 = {accuracy_score(y[va], labels[pv.argmax(1)]):.4f}, "
      f"top-3 = {top_k_accuracy_score(y[va], pv, k=min(3, len(labels)), labels=labels):.4f}")

## 6. Sačuvaj rezultate

Desni panel → *Output* → preuzmi: `splits/`, `labels_grouped.csv`, `embeddings.npy`.
Ili klikni **Save Version** (Save & Run All) — output postaje Kaggle dataset koji sutra dodaješ kao *Input* u notebook za trening.

**Pošalji mi:** izlaz EDA ćelija, izabrani prag + obrazloženje, broj grupa, rezultat linear probe-a.